# Experiments: Advanced Ensemble

In [1]:
from pathlib import Path
from itertools import combinations
import hashlib
import time

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "experiments").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OOF_BASE_DIR = PROJECT_ROOT / "experiments" / "oof_predictions"
TEST_PROB_BASE_DIR = PROJECT_ROOT / "output" / "test_probabilities"
ENS_OOF_DIR = PROJECT_ROOT / "experiments" / "oof_predictions" / "ensemble"
ENS_SUB_DIR = PROJECT_ROOT / "output" / "submissions" / "ensemble"
ENS_REPORT_DIR = PROJECT_ROOT / "reports" / "tables"
ENS_OOF_DIR.mkdir(parents=True, exist_ok=True)
ENS_SUB_DIR.mkdir(parents=True, exist_ok=True)
ENS_REPORT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K_DAILY_SUBMITS = 3
TOP_PER_BACKBONE = 2
GLOBAL_TOP_POOL = 6
GRID_STEP = 0.10
RANDOM_TRIALS = 250
BLEND_MODES = ["arith", "geom"]
TEMP_OPTIONS = [1.0, 0.95, 1.05]

# LB-safe constraints and selection policy
MIN_WEIGHT_2 = 0.20
MAX_WEIGHT_2 = 0.80
MIN_WEIGHT_3 = 0.15
MAX_WEIGHT_3 = 0.60
MIN_DIVERSITY_DIFF_RATE = 0.015
BALANCED_OOF_GAP = 0.003
TOP_EXPLORE_POOL = 120

print(f"OOF_BASE_DIR: {OOF_BASE_DIR}")
print(f"TEST_PROB_BASE_DIR: {TEST_PROB_BASE_DIR}")
print("Weight constraints:")
print(f" 2-model: min={MIN_WEIGHT_2}, max={MAX_WEIGHT_2}")
print(f" 3-model: min={MIN_WEIGHT_3}, max={MAX_WEIGHT_3}")
print(f"Diversity min diff rate: {MIN_DIVERSITY_DIFF_RATE}")

OOF_BASE_DIR: c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\experiments\oof_predictions
TEST_PROB_BASE_DIR: c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\output\test_probabilities
Weight constraints:
 2-model: min=0.2, max=0.8
 3-model: min=0.15, max=0.6
Diversity min diff rate: 0.015


In [3]:
# Coverage check: OOF vs test probabilities
all_oof_runs = sorted([p.stem.replace("oof_", "") for p in OOF_BASE_DIR.glob("oof_baseline_*.csv")])
all_test_prob_runs = sorted([p.stem.replace("test_probs_", "") for p in TEST_PROB_BASE_DIR.glob("test_probs_baseline_*.csv")])

missing_test_prob_runs = [rn for rn in all_oof_runs if rn not in set(all_test_prob_runs)]

print(f"Total OOF runs: {len(all_oof_runs)}")
print(f"Total test_probs runs: {len(all_test_prob_runs)}")
print(f"Missing test_probs runs: {len(missing_test_prob_runs)}")

if len(missing_test_prob_runs) > 0:
    print("Run yang perlu generate test_probs (tanpa retrain jika checkpoint ada):")
    for i, rn in enumerate(missing_test_prob_runs, start=1):
        print(f"{i}. {rn}")

Total OOF runs: 12
Total test_probs runs: 12
Missing test_probs runs: 0


In [4]:
def infer_backbone(run_name: str):
    rn = run_name.lower()
    if "resnext50_32x4d" in rn:
        return "resnext50_32x4d"
    if "efficientnet_b3" in rn:
        return "efficientnet_b3"
    if "convnext_tiny" in rn:
        return "convnext_tiny"
    return "unknown"

def load_oof(run_name: str):
    p = OOF_BASE_DIR / f"oof_{run_name}.csv"
    if not p.exists():
        raise FileNotFoundError(f"OOF file tidak ditemukan: {p}")
    df = pd.read_csv(p).sort_values("row_id").reset_index(drop=True)
    prob_cols = sorted([c for c in df.columns if c.startswith("prob_")])
    if len(prob_cols) == 0:
        raise ValueError(f"Kolom probabilitas tidak ditemukan di: {p}")
    return df, prob_cols

def load_test_probs(run_name: str):
    p = TEST_PROB_BASE_DIR / f"test_probs_{run_name}.csv"
    if not p.exists():
        raise FileNotFoundError(f"Test probability file tidak ditemukan: {p}")
    df = pd.read_csv(p).sort_values("id").reset_index(drop=True)
    prob_cols = sorted([c for c in df.columns if c.startswith("prob_")])
    if len(prob_cols) == 0:
        raise ValueError(f"Kolom probabilitas test tidak ditemukan di: {p}")
    return df, prob_cols

def quick_oof_macro_f1(df_oof: pd.DataFrame, prob_cols: list[str]):
    y_true = df_oof["label"].values
    label_names = [c.replace("prob_", "") for c in prob_cols]
    y_pred = np.array(label_names)[df_oof[prob_cols].values.argmax(axis=1)]
    return float(f1_score(y_true, y_pred, average="macro"))

candidate_rows = []
for p in sorted(OOF_BASE_DIR.glob("oof_baseline_*.csv")):
    run_name = p.stem.replace("oof_", "")
    tp = TEST_PROB_BASE_DIR / f"test_probs_{run_name}.csv"
    if not tp.exists():
        continue
    df_oof, prob_cols = load_oof(run_name)
    candidate_rows.append({
        "run_name": run_name,
        "backbone": infer_backbone(run_name),
        "oof_macro_f1": quick_oof_macro_f1(df_oof, prob_cols),
    })

if len(candidate_rows) < 3:
    raise RuntimeError("Butuh minimal 3 run dengan pasangan OOF + test_probs.")

candidate_df = pd.DataFrame(candidate_rows).sort_values("oof_macro_f1", ascending=False).reset_index(drop=True)
display(candidate_df.head(20))

,run_name,backbone,oof_macro_f1
0,baseline_convnext_tiny_img288_20260331_232502,convnext_tiny,0.921442
1,baseline_resnext50_32x4d_img288_20260330_233148,resnext50_32x4d,0.916101
2,baseline_efficientnet_b3_img288_20260328_145819,efficientnet_b3,0.915835
3,baseline_efficientnet_b3_img288_20260328_213015,efficientnet_b3,0.914914
4,baseline_convnext_tiny_img320_20260328_233846,convnext_tiny,0.912581
5,baseline_efficientnet_b3_img288_20260328_172656,efficientnet_b3,0.911548
6,baseline_convnext_tiny_img288_20260328_142211,convnext_tiny,0.908282
7,baseline_convnext_tiny_img320_20260331_224553,convnext_tiny,0.901674
8,baseline_resnext50_32x4d_img224_20260328_222704,resnext50_32x4d,0.900436
9,baseline_convnext_tiny_img224_20260328_134450,convnext_tiny,0.900184


In [5]:
# Build candidate run pool: strongest + diversity backbone
pool_rows = []
for bb, g in candidate_df.groupby("backbone", dropna=False):
    pool_rows.append(g.head(TOP_PER_BACKBONE))
pool_df = pd.concat(pool_rows, axis=0, ignore_index=True)
pool_df = pd.concat([pool_df, candidate_df.head(GLOBAL_TOP_POOL)], axis=0, ignore_index=True)
pool_df = pool_df.drop_duplicates(subset=["run_name"]).sort_values("oof_macro_f1", ascending=False).reset_index(drop=True)

if len(pool_df) < 3:
    raise RuntimeError("Pool run kurang dari 3 setelah filtering.")

run_pool = pool_df["run_name"].tolist()
print(f"Run pool ({len(run_pool)}):")
for i, rn in enumerate(run_pool, start=1):
    print(f"{i}. {rn}")
display(pool_df)

# Preload artifacts once
oof_map = {}
test_map = {}
prob_cols_ref = None
label_names = None
base_y = None
base_ids = None

for rn in run_pool:
    df_oof, prob_cols = load_oof(rn)
    df_test, prob_cols_test = load_test_probs(rn)
    if prob_cols != prob_cols_test:
        raise ValueError(f"Prob column mismatch OOF vs test: {rn}")

    if prob_cols_ref is None:
        prob_cols_ref = prob_cols
        label_names = [c.replace("prob_", "") for c in prob_cols_ref]
        base_y = df_oof["label"].values
        base_ids = df_test["id"].tolist()
    else:
        if prob_cols != prob_cols_ref:
            raise ValueError(f"Prob column mismatch antar run: {rn}")
        if not np.array_equal(base_y, df_oof["label"].values):
            raise ValueError(f"OOF label order mismatch: {rn}")
        if base_ids != df_test["id"].tolist():
            raise ValueError(f"Test id order mismatch: {rn}")

    oof_map[rn] = df_oof[prob_cols_ref].values.astype(np.float32)
    test_map[rn] = df_test[prob_cols_ref].values.astype(np.float32)

print(f"Loaded artifacts for {len(run_pool)} runs")

Run pool (7):
1. baseline_convnext_tiny_img288_20260331_232502
2. baseline_resnext50_32x4d_img288_20260330_233148
3. baseline_efficientnet_b3_img288_20260328_145819
4. baseline_efficientnet_b3_img288_20260328_213015
5. baseline_convnext_tiny_img320_20260328_233846
6. baseline_efficientnet_b3_img288_20260328_172656
7. baseline_resnext50_32x4d_img224_20260328_222704


,run_name,backbone,oof_macro_f1
0,baseline_convnext_tiny_img288_20260331_232502,convnext_tiny,0.921442
1,baseline_resnext50_32x4d_img288_20260330_233148,resnext50_32x4d,0.916101
2,baseline_efficientnet_b3_img288_20260328_145819,efficientnet_b3,0.915835
3,baseline_efficientnet_b3_img288_20260328_213015,efficientnet_b3,0.914914
4,baseline_convnext_tiny_img320_20260328_233846,convnext_tiny,0.912581
5,baseline_efficientnet_b3_img288_20260328_172656,efficientnet_b3,0.911548
6,baseline_resnext50_32x4d_img224_20260328_222704,resnext50_32x4d,0.900436


Loaded artifacts for 7 runs


In [6]:
def softmax_np(x: np.ndarray):
    x = x - x.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / ex.sum(axis=1, keepdims=True)

def apply_temperature(prob: np.ndarray, temp: float, eps: float = 1e-12):
    p = np.clip(prob, eps, 1.0)
    logits = np.log(p) / max(temp, 1e-6)
    return softmax_np(logits)

def blend_probs(prob_list: list[np.ndarray], weights: np.ndarray, mode: str):
    w = weights / weights.sum()
    if mode == "arith":
        out = np.zeros_like(prob_list[0], dtype=np.float64)
        for i, p in enumerate(prob_list):
            out += w[i] * p
        out = out / out.sum(axis=1, keepdims=True)
        return out.astype(np.float32)
    if mode == "geom":
        logp = np.zeros_like(prob_list[0], dtype=np.float64)
        for i, p in enumerate(prob_list):
            logp += w[i] * np.log(np.clip(p, 1e-12, 1.0))
        out = np.exp(logp)
        out = out / out.sum(axis=1, keepdims=True)
        return out.astype(np.float32)
    raise ValueError(f"Unknown mode: {mode}")

def grid_weights(n: int, step: float):
    vals = np.round(np.arange(0.0, 1.0 + step, step), 4)
    out = []
    if n == 2:
        for a in vals:
            b = round(1.0 - a, 4)
            if b < 0 or b > 1:
                continue
            out.append(np.array([a, b], dtype=np.float32))
    elif n == 3:
        for a in vals:
            for b in vals:
                c = round(1.0 - a - b, 4)
                if c < 0 or c > 1:
                    continue
                out.append(np.array([a, b, c], dtype=np.float32))
    return out

def random_weights(n: int, trials: int, seed: int = 42):
    rng = np.random.default_rng(seed)
    return [rng.dirichlet(np.ones(n)).astype(np.float32) for _ in range(trials)]

def submission_fingerprint(labels: np.ndarray):
    s = "|".join(labels.tolist())
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def predictions_diff_rate(a: np.ndarray, b: np.ndarray):
    return float(np.mean(a != b))

def is_weight_valid(w: np.ndarray):
    n = len(w)
    if n == 2:
        return bool((w.min() >= MIN_WEIGHT_2) and (w.max() <= MAX_WEIGHT_2))
    if n == 3:
        return bool((w.min() >= MIN_WEIGHT_3) and (w.max() <= MAX_WEIGHT_3))
    return True

experiment_rows = []
all_run_combos = []
for k in [2, 3]:
    all_run_combos.extend(list(combinations(run_pool, k)))

print(f"Evaluating {len(all_run_combos)} run combinations")

for combo in all_run_combos:
    n = len(combo)
    gw = grid_weights(n=n, step=GRID_STEP)
    rw = random_weights(n=n, trials=RANDOM_TRIALS, seed=42 + n)
    weights_list = gw + rw

    oof_prob_list_base = [oof_map[rn] for rn in combo]
    test_prob_list_base = [test_map[rn] for rn in combo]

    for temp in TEMP_OPTIONS:
        oof_prob_list = [apply_temperature(p, temp) for p in oof_prob_list_base]
        test_prob_list = [apply_temperature(p, temp) for p in test_prob_list_base]

        for mode in BLEND_MODES:
            seen_weight = set()
            for w in weights_list:
                wn = w / w.sum()
                key = tuple(np.round(wn, 4).tolist())
                if key in seen_weight:
                    continue
                seen_weight.add(key)

                if not is_weight_valid(wn):
                    continue

                oof_blend = blend_probs(oof_prob_list, wn, mode=mode)
                oof_pred_idx = oof_blend.argmax(axis=1)
                oof_pred = np.array(label_names)[oof_pred_idx]
                oof_f1 = float(f1_score(base_y, oof_pred, average="macro"))

                test_blend = blend_probs(test_prob_list, wn, mode=mode)
                test_pred_idx = test_blend.argmax(axis=1).astype(np.int16)
                test_pred = np.array(label_names)[test_pred_idx]
                fp = submission_fingerprint(test_pred)

                weight_map = {combo[i]: round(float(wn[i]), 4) for i in range(n)}
                experiment_rows.append({
                    "run_combo": " | ".join(combo),
                    "n_models": n,
                    "blend_mode": mode,
                    "temperature": float(temp),
                    "weights": str(weight_map),
                    "oof_macro_f1": oof_f1,
                    "submission_fp": fp,
                    "test_pred_idx": test_pred_idx,
                    "combo": combo,
                    "weight_vec": wn,
                    "max_weight": float(wn.max()),
                    "min_weight": float(wn.min()),
                })

exp_df = pd.DataFrame(experiment_rows).sort_values("oof_macro_f1", ascending=False).reset_index(drop=True)
print(f"Total evaluated candidates (after weight constraints): {len(exp_df)}")
display(exp_df[["n_models", "blend_mode", "temperature", "oof_macro_f1", "max_weight", "min_weight", "weights"]].head(20))

Evaluating 56 run combinations
Total evaluated candidates (after weight constraints): 39438


,n_models,blend_mode,temperature,oof_macro_f1,max_weight,min_weight,weights
0,3,arith,1.00,0.932330,0.515005,0.225648,{'baseline_convnext_tiny_img288_20260331_23250...
1,3,arith,0.95,0.932330,0.515005,0.225648,{'baseline_convnext_tiny_img288_20260331_23250...
2,3,arith,0.95,0.932221,0.400000,0.200000,{'baseline_convnext_tiny_img288_20260331_23250...
3,3,arith,1.05,0.932221,0.400000,0.200000,{'baseline_convnext_tiny_img288_20260331_23250...
4,3,arith,1.00,0.932221,0.400000,0.200000,{'baseline_convnext_tiny_img288_20260331_23250...
5,3,arith,0.95,0.931847,0.433224,0.227694,{'baseline_convnext_tiny_img288_20260331_23250...
6,3,arith,0.95,0.931818,0.500000,0.200000,{'baseline_convnext_tiny_img288_20260331_23250...
7,3,arith,1.00,0.931818,0.500000,0.200000,{'baseline_convnext_tiny_img288_20260331_23250...
8,3,arith,0.95,0.931818,0.479097,0.205986,{'baseline_convnext_tiny_img288_20260331_23250...
9,3,arith,1.05,0.931818,0.515005,0.225648,{'baseline_convnext_tiny_img288_20260331_23250...


In [7]:
# Multi-objective daily submit plan: exploit + balanced + explore
if len(exp_df) == 0:
    raise RuntimeError("Tidak ada kandidat ensemble setelah filtering.")

def row_diff_rate(row_a: pd.Series, row_b: pd.Series):
    return predictions_diff_rate(row_a["test_pred_idx"], row_b["test_pred_idx"])

def pick_exploit(df: pd.DataFrame):
    return df.iloc[0]

def pick_balanced(df: pd.DataFrame, exploit_row: pd.Series):
    top_oof = float(df.iloc[0]["oof_macro_f1"])
    cand = df[df["oof_macro_f1"] >= top_oof - BALANCED_OOF_GAP].copy()
    if len(cand) == 0:
        cand = df.head(min(len(df), TOP_EXPLORE_POOL)).copy()

    cand = cand[cand["submission_fp"] != exploit_row["submission_fp"]].copy()
    if len(cand) == 0:
        return None

    cand["div_to_exploit"] = cand.apply(lambda r: row_diff_rate(r, exploit_row), axis=1)
    cand = cand[cand["div_to_exploit"] >= MIN_DIVERSITY_DIFF_RATE].copy()
    if len(cand) == 0:
        return None

    # prioritize diversity while keeping OOF close to top
    cand["balanced_score"] = cand["oof_macro_f1"] + 0.5 * cand["div_to_exploit"]
    cand = cand.sort_values(["balanced_score", "oof_macro_f1"], ascending=False).reset_index(drop=True)
    return cand.iloc[0]

def pick_explore(df: pd.DataFrame, chosen_rows: list[pd.Series]):
    chosen_fp = {r["submission_fp"] for r in chosen_rows}
    base = df.head(min(len(df), TOP_EXPLORE_POOL)).copy()
    base = base[~base["submission_fp"].isin(chosen_fp)].copy()
    if len(base) == 0:
        return None

    # Encourage different blend recipe from exploit candidate
    exploit = chosen_rows[0]
    base["is_recipe_diff"] = (
        (base["blend_mode"] != exploit["blend_mode"]) |
        (base["temperature"] != exploit["temperature"]) |
        (base["n_models"] != exploit["n_models"])
    ).astype(int)

    def min_div_to_chosen(row):
        return min(row_diff_rate(row, c) for c in chosen_rows)

    base["min_div_to_chosen"] = base.apply(min_div_to_chosen, axis=1)
    base = base[base["min_div_to_chosen"] >= MIN_DIVERSITY_DIFF_RATE].copy()
    if len(base) == 0:
        return None

    base["explore_score"] = (
        0.7 * base["oof_macro_f1"] +
        0.3 * base["min_div_to_chosen"] +
        0.0005 * base["is_recipe_diff"]
    )
    base = base.sort_values(["explore_score", "oof_macro_f1"], ascending=False).reset_index(drop=True)
    return base.iloc[0]

selected_with_role = []

exploit_row = pick_exploit(exp_df)
selected_with_role.append(("exploit", exploit_row))

balanced_row = pick_balanced(exp_df, exploit_row)
if balanced_row is not None:
    selected_with_role.append(("balanced", balanced_row))

explore_row = pick_explore(exp_df, [r for _, r in selected_with_role])
if explore_row is not None:
    selected_with_role.append(("explore", explore_row))

# Fill remaining slots by highest OOF under diversity constraints
if len(selected_with_role) < TOP_K_DAILY_SUBMITS:
    chosen_rows_only = [r for _, r in selected_with_role]
    chosen_fp = {r["submission_fp"] for r in chosen_rows_only}
    for _, row in exp_df.iterrows():
        if len(selected_with_role) >= TOP_K_DAILY_SUBMITS:
            break
        if row["submission_fp"] in chosen_fp:
            continue
        if len(chosen_rows_only) > 0:
            min_div = min(row_diff_rate(row, c) for c in chosen_rows_only)
            if min_div < MIN_DIVERSITY_DIFF_RATE:
                continue
        selected_with_role.append(("fallback", row))
        chosen_rows_only = [r for _, r in selected_with_role]
        chosen_fp.add(row["submission_fp"])

if len(selected_with_role) == 0:
    raise RuntimeError("Tidak ada kandidat submit setelah seleksi multi-objective.")

timestamp = time.strftime("%Y%m%d_%H%M%S")
submit_plan_rows = []

for rank, (role, row) in enumerate(selected_with_role, start=1):
    combo = list(row["combo"])
    wn = row["weight_vec"]
    mode = row["blend_mode"]
    temp = float(row["temperature"])

    # rebuild blended probs for export consistency
    oof_prob_list = [apply_temperature(oof_map[rn], temp) for rn in combo]
    test_prob_list = [apply_temperature(test_map[rn], temp) for rn in combo]
    oof_blend = blend_probs(oof_prob_list, wn, mode=mode)
    test_blend = blend_probs(test_prob_list, wn, mode=mode)
    oof_pred = np.array(label_names)[oof_blend.argmax(axis=1)]
    test_pred = np.array(label_names)[test_blend.argmax(axis=1)]
    oof_f1 = float(f1_score(base_y, oof_pred, average="macro"))

    weight_map = {combo[i]: round(float(wn[i]), 4) for i in range(len(combo))}
    blend_name_parts = [f"{rn}-{str(weight_map[rn]).replace('.', 'p')}" for rn in combo]
    blend_name = f"exp_{role}_{mode}_t{str(temp).replace('.', 'p')}_{'__'.join(blend_name_parts)}"

    oof_df_anchor, _ = load_oof(combo[0])
    oof_out = oof_df_anchor[["row_id", "file_path", "label", "fold"]].copy()
    oof_out["pred_label"] = oof_pred
    oof_out[prob_cols_ref] = oof_blend
    oof_path = ENS_OOF_DIR / f"oof_{blend_name}_{timestamp}.csv"
    oof_out.to_csv(oof_path, index=False)

    sub_out = pd.DataFrame({"id": base_ids, "label": test_pred})
    sub_path = ENS_SUB_DIR / f"submission_{blend_name}_{timestamp}.csv"
    sub_out.to_csv(sub_path, index=False)

    # diversity summary wrt previous selections
    diversity_to_previous = np.nan
    if rank > 1:
        prev_rows = [r for _, r in selected_with_role[:rank - 1]]
        diversity_to_previous = float(min(row_diff_rate(row, pr) for pr in prev_rows))

    submit_plan_rows.append({
        "rank": rank,
        "role": role,
        "ensemble_name": blend_name,
        "n_models": len(combo),
        "blend_mode": mode,
        "temperature": temp,
        "oof_macro_f1": oof_f1,
        "weights": str(weight_map),
        "max_weight": float(np.max(wn)),
        "min_weight": float(np.min(wn)),
        "submission_fp": row["submission_fp"],
        "min_div_to_previous": diversity_to_previous,
        "oof_file": str(oof_path),
        "submission_file": str(sub_path),
    })

submit_plan_df = pd.DataFrame(submit_plan_rows).sort_values("rank", ascending=True).reset_index(drop=True)
display(submit_plan_df)

full_report_path = ENS_REPORT_DIR / f"experiment_candidates_full_{timestamp}.csv"
plan_report_path = ENS_REPORT_DIR / f"experiment_submit_plan_{timestamp}.csv"
exp_df.drop(columns=["test_pred_idx", "combo", "weight_vec"], errors="ignore").to_csv(full_report_path, index=False)
submit_plan_df.to_csv(plan_report_path, index=False)
print(f"Saved full report: {full_report_path}")
print(f"Saved submit plan: {plan_report_path}")

,rank,role,ensemble_name,n_models,blend_mode,temperature,oof_macro_f1,weights,max_weight,min_weight,submission_fp,min_div_to_previous,oof_file,submission_file
0,1,exploit,exp_exploit_arith_t1p0_baseline_convnext_tiny_...,3,arith,1.00,0.932330,{'baseline_convnext_tiny_img288_20260331_23250...,0.515005,0.225648,cb98e2afc59b5495ebc831b5ce3ef5aa,NaN,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...
1,2,balanced,exp_balanced_arith_t1p0_baseline_convnext_tiny...,3,arith,1.00,0.929653,{'baseline_convnext_tiny_img288_20260331_23250...,0.484200,0.156738,d317f87dd548d4e8d2becc3413b184d6,0.037129,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...
2,3,explore,exp_explore_arith_t1p05_baseline_resnext50_32x...,3,arith,1.05,0.931125,{'baseline_resnext50_32x4d_img288_20260330_233...,0.400000,0.300000,d84751048fca2efcfac4031bca2e4da8,0.019802,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...


Saved full report: c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\reports\tables\experiment_candidates_full_20260401_020410.csv
Saved submit plan: c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\reports\tables\experiment_submit_plan_20260401_020410.csv
